# Strategy Backtesting

Backtest combined ML + scalping strategy.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import sys

sys.path.insert(0, str(Path.cwd().parent))

from src.data_collection.load_kaggle_data import load_kaggle_data
from src.preprocessing.clean_data import clean_ohlcv_data
from src.utils.data_split import split_data_by_date
from src.utils.config import DEFAULT_TICKERS, INITIAL_CAPITAL, COMMISSION_RATE

print("="*70)
print("BACKTESTING ON 2024 DATA")
print("="*70)
print(f"Initial Capital: ${INITIAL_CAPITAL:,.0f}")
print(f"Commission Rate: {COMMISSION_RATE*100:.2f}%")
print("="*70)

## Check Available Data Intervals

In [ ]:
class SimpleBacktester:
    """Simple backtester for strategy evaluation"""
    
    def __init__(self, initial_capital=100000, commission=0.0005):
        self.initial_capital = initial_capital
        self.commission = commission
        self.reset()
    
    def reset(self):
        self.capital = self.initial_capital
        self.position = False
        self.entry_price = None
        self.trades = []
        self.equity_curve = []
    
    def backtest(self, data, signal_col='signal'):
        """Run backtest"""
        self.reset()
        data = data.copy()
        
        for i in range(1, len(data)):
            row = data.iloc[i]
            prev = data.iloc[i-1]
            
            price = row['Close']
            signal = prev[signal_col]  # Shift to avoid lookahead bias
            
            # Entry
            if not self.position and signal == 1:
                qty = int(self.capital / price)
                if qty > 0:
                    cost = qty * price * (1 + self.commission)
                    if cost <= self.capital:
                        self.capital -= cost
                        self.position = True
                        self.entry_price = price
            
            # Exit
            elif self.position and signal == -1:
                proceeds = int(self.capital / self.entry_price) * price * (1 - self.commission)
                pnl = proceeds - self.capital
                self.capital = self.capital + pnl
                self.trades.append(pnl)
                self.position = False
            
            # Equity update
            if self.position:
                equity = self.capital + int(self.capital / self.entry_price) * price
            else:
                equity = self.capital
            
            self.equity_curve.append(equity)
        
        return self.metrics()
    
    def metrics(self):
        """Calculate performance metrics"""
        equity = pd.Series(self.equity_curve)
        
        total_return = (equity.iloc[-1] - self.initial_capital) / self.initial_capital
        
        # Sharpe ratio
        returns = equity.pct_change().dropna()
        sharpe = (returns.mean() / returns.std()) * np.sqrt(252) if returns.std() > 0 else 0
        
        # Max drawdown
        cummax = equity.cummax()
        drawdown = (equity - cummax) / cummax
        max_dd = drawdown.min()
        
        # Win rate
        wins = sum(1 for p in self.trades if p > 0)
        win_rate = wins / len(self.trades) if self.trades else 0
        
        return {
            'final_equity': equity.iloc[-1],
            'total_return': total_return,
            'sharpe_ratio': sharpe,
            'max_drawdown': max_dd,
            'total_trades': len(self.trades),
            'win_rate': win_rate,
            'equity_curve': equity
        }

In [ ]:
def add_scalping_signals(data):
    """Add scalping signals"""
    df = data.copy()
    
    # RSI
    delta = df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = -delta.where(delta < 0, 0).rolling(14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # SMA
    df['SMA_20'] = df['Close'].rolling(20).mean()
    df['SMA_50'] = df['Close'].rolling(50).mean()
    df['Volume_SMA'] = df['Volume'].rolling(20).mean()
    
    # Signals
    buy = ((df['RSI'] < 30) | ((df['Close'] > df['SMA_20']) & 
           (df['SMA_20'] > df['SMA_50']) & (df['Volume'] > df['Volume_SMA']))).astype(int)
    sell = ((df['RSI'] > 70) | ((df['Close'] < df['SMA_20']) & 
            (df['SMA_20'] < df['SMA_50']))).astype(int) * -1
    
    df['signal'] = (buy + sell).clip(-1, 1)
    return df

# Backtest all tickers on 2024 data
results = {}

print("\nBacktesting all tickers on 2024 data...")
print("="*70)

for ticker in DEFAULT_TICKERS:
    print(f"\n{ticker}:")
    
    try:
        raw = load_kaggle_data(ticker)
        clean = clean_ohlcv_data(raw)
        train, test = split_data_by_date(clean)
        
        test_signals = add_scalping_signals(test.copy())
        
        bt = SimpleBacktester(INITIAL_CAPITAL, COMMISSION_RATE)
        metrics = bt.backtest(test_signals, 'signal')
        
        results[ticker] = metrics
        
        print(f"  Final Equity: ${metrics['final_equity']:,.0f}")
        print(f"  Return: {metrics['total_return']*100:+.2f}%")
        print(f"  Sharpe: {metrics['sharpe_ratio']:.4f}")
        print(f"  Max DD: {metrics['max_drawdown']*100:.2f}%")
        print(f"  Trades: {metrics['total_trades']} | Win Rate: {metrics['win_rate']*100:.1f}%")
        
    except Exception as e:
        print(f"  ❌ Error: {str(e)[:60]}")

print("\n" + "="*70)

In [ ]:
# Create summary table
summary = pd.DataFrame({
    'Return (%)': [v['total_return']*100 for v in results.values()],
    'Sharpe': [v['sharpe_ratio'] for v in results.values()],
    'Win Rate (%)': [v['win_rate']*100 for v in results.values()],
    'Max DD (%)': [v['max_drawdown']*100 for v in results.values()],
    'Trades': [v['total_trades'] for v in results.values()],
}, index=results.keys())

print("\nBacktest Summary (2024):")
print(summary.to_string())

print(f"\nAverage Return: {summary['Return (%)'].mean():.2f}%")
print(f"Average Sharpe: {summary['Sharpe'].mean():.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Returns
ax = axes[0, 0]
returns = [results[t]['total_return']*100 for t in DEFAULT_TICKERS]
colors = ['green' if r > 0 else 'red' for r in returns]
ax.bar(range(len(DEFAULT_TICKERS)), returns, color=colors)
ax.set_xticks(range(len(DEFAULT_TICKERS)))
ax.set_xticklabels(DEFAULT_TICKERS, rotation=45)
ax.set_title('2024 Returns by Ticker')
ax.set_ylabel('Return (%)')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

# Sharpe Ratio
ax = axes[0, 1]
sharpes = [results[t]['sharpe_ratio'] for t in DEFAULT_TICKERS]
colors = ['green' if s > 0 else 'red' for s in sharpes]
ax.bar(range(len(DEFAULT_TICKERS)), sharpes, color=colors)
ax.set_xticks(range(len(DEFAULT_TICKERS)))
ax.set_xticklabels(DEFAULT_TICKERS, rotation=45)
ax.set_title('Sharpe Ratio by Ticker')
ax.set_ylabel('Sharpe Ratio')
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax.grid(True, alpha=0.3, axis='y')

# Win Rate
ax = axes[1, 0]
win_rates = [results[t]['win_rate']*100 for t in DEFAULT_TICKERS]
ax.bar(range(len(DEFAULT_TICKERS)), win_rates, color='steelblue')
ax.set_xticks(range(len(DEFAULT_TICKERS)))
ax.set_xticklabels(DEFAULT_TICKERS, rotation=45)
ax.set_title('Win Rate by Ticker')
ax.set_ylabel('Win Rate (%)')
ax.axhline(y=50, color='red', linestyle='--', alpha=0.5, label='50% (Break-even)')
ax.set_ylim([0, 100])
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Equity Curves (First 3 tickers)
ax = axes[1, 1]
for ticker in DEFAULT_TICKERS[:3]:
    equity = results[ticker]['equity_curve']
    ax.plot(range(len(equity)), equity, label=ticker, linewidth=1)
ax.set_title('Equity Curve (First 3 Tickers)')
ax.set_xlabel('Trading Days (2024)')
ax.set_ylabel('Equity ($)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare strategy with buy-and-hold for each ticker
print("\n" + "="*70)
print("STRATEGY vs BUY-AND-HOLD COMPARISON")
print("="*70)

comparison = []

for ticker in DEFAULT_TICKERS:
    if ticker not in results:
        continue
    
    try:
        # Get test data for this ticker
        raw = load_kaggle_data(ticker)
        clean = clean_ohlcv_data(raw)
        _, test = split_data_by_date(clean)
        
        # Calculate buy-and-hold return
        buy_hold_return = (test['Close'].iloc[-1] - test['Close'].iloc[0]) / test['Close'].iloc[0]
        
        # Strategy return
        strategy_return = results[ticker]['total_return']
        
        # Outperformance
        outperformance = strategy_return - buy_hold_return
        
        comparison.append({
            'Ticker': ticker,
            'Buy-Hold Return (%)': buy_hold_return * 100,
            'Strategy Return (%)': strategy_return * 100,
            'Outperformance (%)': outperformance * 100
        })
        
        print(f"\n{ticker}:")
        print(f"  Buy-and-Hold:  {buy_hold_return:+.2%}")
        print(f"  Strategy:      {strategy_return:+.2%}")
        print(f"  Outperformance: {outperformance:+.2%}")
        
    except Exception as e:
        print(f"\n{ticker}: Error - {str(e)[:50]}")

if comparison:
    comp_df = pd.DataFrame(comparison)
    print("\n" + "="*70)
    print("Summary Comparison Table:")
    print(comp_df.to_string(index=False))
    print(f"\nAverage Strategy Outperformance: {comp_df['Outperformance (%)'].mean():.2f}%")